# model.ipynb

DeepLabV3+ dengan encoder ResNet50 pretrained untuk segmentasi biner manusia.

Properti utama yang tertanam:

1. **Head output float32.** Konv 1x1 terakhir dan sigmoid dipaku ke `dtype="float32"`, sehingga peta probabilitas berpresisi penuh di bawah `mixed_float16` (di Keras 2 maupun Keras 3).
2. **Bobot yang dapat dikonfigurasi dan backbone beku opsional** melalui `deeplabv3_plus(shape, weights, freeze_backbone)`.
3. **Fine-tuning dua fase yang aman** tanpa pelacakan variabel ganda: layer backbone ditandai dengan boolean `_is_backbone` (dihitung dari nama layer encoder) dan `set_backbone_trainable` mengubahnya, sambil menjaga BatchNorm backbone tetap beku.

> Karena tap terdalam adalah `conv4_block6_out`, Keras memangkas tahap `conv5` yang tidak digunakan dan model yang dirakit memiliki sekitar 10,9 juta parameter, bukan 23,5 juta.

Muat dengan `%run model.ipynb`.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
from tensorflow.keras.layers import (
    Conv2D, DepthwiseConv2D, BatchNormalization, Activation,
    UpSampling2D, Concatenate, Input,
    AveragePooling2D, GlobalAveragePooling2D, Reshape, Dense, Dropout,
)
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50  # type: ignore

In [ ]:
try:
    _DEMO
except NameError:
    _DEMO = True  # True saat notebook ini dijalankan sendiri; disetel False oleh pemanggil %run

### Blok pembangun

Konvolusi separable depthwise (pencampuran spasial lalu pencampuran kanal) dan atensi kanal Squeeze-and-Excitation.

In [ ]:
def separable_conv_block(inputs, filters, dilation_rate=1):
    """Konvolusi separable depthwise: pencampuran spasial lalu pencampuran kanal."""
    x = DepthwiseConv2D(
        3, padding="same", use_bias=False, dilation_rate=dilation_rate
    )(inputs)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(filters, 1, padding="same", use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def SqueezeAndExcite(inputs, ratio=8):
    """Atensi kanal melalui global average pooling dan gating sigmoid."""
    init = inputs
    filters = init.shape[-1]
    se = GlobalAveragePooling2D()(init)
    se = Reshape((1, 1, filters))(se)
    se = Dense(
        filters // ratio,
        activation="relu",
        kernel_initializer="he_normal",
        use_bias=False,
    )(se)
    se = Dense(
        filters,
        activation="sigmoid",
        kernel_initializer="he_normal",
        use_bias=False,
    )(se)
    return init * se

### ASPP

Enam cabang konteks paralel (global pool, konv 1x1, dan konv separable terdilasi pada rate 6/12/18/24), dikonkatenasi, diproyeksikan ke 256, lalu Dropout.

In [ ]:
def ASPP(inputs):
    """
    Atrous Spatial Pyramid Pooling - enam cabang konteks paralel.

    Cabang 1 : global image pooling (konteks seluruh gambar)
    Cabang 2 : konv 1x1 (pointwise / lokal)
    Cabang 3 : konv separable terdilasi, rate 6
    Cabang 4 : konv separable terdilasi, rate 12
    Cabang 5 : konv separable terdilasi, rate 18
    Cabang 6 : konv separable terdilasi, rate 24

    Semua cabang dikonkatenasi, diproyeksikan ke 256 kanal, lalu Dropout(0.3).
    """
    shape = inputs.shape

    y1 = AveragePooling2D(pool_size=(shape[1], shape[2]))(inputs)
    y1 = Conv2D(256, 1, padding="same", use_bias=False)(y1)
    y1 = BatchNormalization()(y1)
    y1 = Activation("relu")(y1)
    y1 = UpSampling2D((shape[1], shape[2]), interpolation="bilinear")(y1)

    y2 = Conv2D(256, 1, padding="same", use_bias=False)(inputs)
    y2 = BatchNormalization()(y2)
    y2 = Activation("relu")(y2)

    y3 = separable_conv_block(inputs, 256, dilation_rate=6)
    y4 = separable_conv_block(inputs, 256, dilation_rate=12)
    y5 = separable_conv_block(inputs, 256, dilation_rate=18)
    y6 = separable_conv_block(inputs, 256, dilation_rate=24)

    y = Concatenate()([y1, y2, y3, y4, y5, y6])
    y = Conv2D(256, 1, padding="same", use_bias=False)(y)
    y = BatchNormalization()(y)
    y = Activation("relu")(y)
    y = Dropout(0.3)(y)
    return y

### Helper pembekuan backbone

Bekukan atau cairkan backbone ResNet50 menggunakan tag `_is_backbone`, sambil menjaga BatchNorm backbone tetap beku untuk stabilitas batch kecil. Kompilasi ulang model setelah mengubah trainability.

In [ ]:
def set_backbone_trainable(model, trainable, freeze_bn=True):
    """
    Bekukan atau cairkan backbone ResNet50 dari model yang dibangun oleh deeplabv3_plus.

    Mengandalkan tag boolean _is_backbone yang disetel saat konstruksi model,
    sehingga tidak pernah menyentuh head decoder/ASPP. Ketika freeze_bn True,
    layer BatchNormalization backbone dijaga tetap beku bahkan saat sisa
    backbone dicairkan; dengan ukuran batch 2, statistik running jauh lebih
    andal daripada statistik per-batch, sehingga ini menstabilkan fine-tuning.

    Panggil model.compile(...) lagi setelah mengubah trainability.
    """
    for layer in model.layers:
        if getattr(layer, "_is_backbone", False):
            if freeze_bn and isinstance(layer, BatchNormalization):
                layer.trainable = False
            else:
                layer.trainable = trainable
    return model

### Model DeepLabV3+

Encoder ResNet50 dengan tiga tap (`conv1_relu`, `conv2_block3_out`, `conv4_block6_out`), ASPP, decoder SE tiga tahap, dan head output float32.

In [ ]:
def deeplabv3_plus(shape, weights="imagenet", freeze_backbone=False):
    """
    Membangun DeepLabV3+ dengan encoder ResNet50.

    Parameter
    ----------
    shape : tuple
        Bentuk input, contoh (512, 512, 3).
    weights : "imagenet" | None
        Inisialisasi encoder. "imagenet" mengunduh bobot pretrained pertama
        kali; None menginisialisasi secara acak (berguna untuk smoke test offline).
    freeze_backbone : bool
        Jika True, encoder dibekukan tepat setelah konstruksi (fase warmup).
        Layer BatchNormalization di backbone selalu dibekukan saat build
        agar eksekusi beku dan warmup berbagi statistik yang sama.

    Ukuran feature-map ResNet50 untuk input 512x512:
        conv1_relu       : 256x256 x   64   -> skip1
        conv2_block3_out : 128x128 x  256   -> skip2
        conv4_block6_out :  32x32  x 1024   -> input ASPP (output stride 16)

    Encoder secara efektif adalah ResNet50 hingga conv4_block6_out. Keras memangkas
    tahap conv5 yang tidak digunakan secara otomatis, sehingga model yang dirakit
    memiliki sekitar 10,9 juta parameter, bukan 23,5 juta dari ResNet50 lengkap.
    """
    inputs = Input(shape)

    encoder = ResNet50(weights=weights, include_top=False, input_tensor=inputs)
    # Catat nama layer backbone SEBELUM merakit head agar kita bisa menandai
    # layer model final tanpa mereferensikan objek encoder lagi.
    encoder_layer_names = {layer.name for layer in encoder.layers}

    deep_features = encoder.get_layer("conv4_block6_out").output  # 32x32x1024
    skip2 = encoder.get_layer("conv2_block3_out").output          # 128x128x256
    skip1 = encoder.get_layer("conv1_relu").output                # 256x256x64

    # ASPP
    x = ASPP(deep_features)                                       # 32x32x256

    # Tahap decoder 1: 32x32 -> 128x128
    x = UpSampling2D((4, 4), interpolation="bilinear")(x)
    s2 = Conv2D(48, 1, padding="same", use_bias=False)(skip2)
    s2 = BatchNormalization()(s2)
    s2 = Activation("relu")(s2)
    x = Concatenate()([x, s2])       # 128x128 x 304
    x = SqueezeAndExcite(x)
    x = separable_conv_block(x, 256)
    x = separable_conv_block(x, 256)
    x = SqueezeAndExcite(x)          # 128x128 x 256

    # Tahap decoder 2: 128x128 -> 256x256
    x = UpSampling2D((2, 2), interpolation="bilinear")(x)
    s1 = Conv2D(32, 1, padding="same", use_bias=False)(skip1)
    s1 = BatchNormalization()(s1)
    s1 = Activation("relu")(s1)
    x = Concatenate()([x, s1])       # 256x256 x 288
    x = separable_conv_block(x, 128)
    x = separable_conv_block(x, 64)  # 256x256 x 64

    # Tahap decoder 3: 256x256 -> 512x512
    x = UpSampling2D((2, 2), interpolation="bilinear")(x)
    # Head output float32: paku konv akhir dan sigmoid ke float32 agar
    # peta probabilitas tidak pernah dihasilkan dalam float16 di bawah mixed_float16.
    x = Conv2D(1, 1, padding="same", dtype="float32")(x)
    outputs = Activation("sigmoid", dtype="float32", name="predictions")(x)

    model = Model(inputs, outputs)

    # Tandai layer backbone dengan boolean biasa (tanpa pelacakan variabel).
    for layer in model.layers:
        layer._is_backbone = layer.name in encoder_layer_names

    # BatchNormalization backbone selalu dibekukan (stabilitas batch kecil).
    for layer in model.layers:
        if getattr(layer, "_is_backbone", False) and isinstance(layer, BatchNormalization):
            layer.trainable = False

    if freeze_backbone:
        set_backbone_trainable(model, trainable=False, freeze_bn=True)

    return model

### Self-check

Build offline (bobot acak) dan konfirmasi dtype output serta jumlah parameter freeze/unfreeze.

In [ ]:
if _DEMO:
    # Smoke test offline (bobot acak, tanpa unduhan).
    print("Membangun DeepLabV3+ (bobot acak) untuk self-check...")
    model = deeplabv3_plus((512, 512, 3), weights=None)

    def count_trainable(m):
        return int(sum(tf.keras.backend.count_params(w) for w in m.trainable_weights))

    total = model.count_params()
    print(f"Total parameter           : {total:,}")
    print(f"Dapat dilatih (model penuh): {count_trainable(model):,}")
    print(f"Dtype output              : {model.output.dtype}")

    set_backbone_trainable(model, trainable=False, freeze_bn=True)
    print(f"Dapat dilatih (backbone beku)      : {count_trainable(model):,}")

    set_backbone_trainable(model, trainable=True, freeze_bn=True)
    print(f"Dapat dilatih (backbone aktif, BN beku): {count_trainable(model):,}")
    print(f"Bentuk input  : {model.input_shape}")
    print(f"Bentuk output : {model.output_shape}")